In [ ]:
import pandas as pd
import subprocess
import os
import shlex
import re
import time
from io import StringIO

In [ ]:
# User namme of individual
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')
%env USER_NAME={USER_NAME}

# Main working directory in GCP
my_bucket = os.getenv('WORKSPACE_BUCKET')
MAIN_DIR=f"{my_bucket}/data"
%env MAIN_DIR={MAIN_DIR}

# Master folder for current analysis
SAIGE_DIR="meta/SDoH_MCA_GWAS"
%env SAIGE_DIR={SAIGE_DIR}

# Persistent disk file path to save intermediate files
cur_dir = '/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta'
%env CUR_DIR={cur_dir}

# Phenotype file

## Parameter file

In [ ]:
# Initialize an Empty DataFrame
param = []

for folder_name, phenocol, sexfilter, mca in [
    ("asthma", "asthma", 3, "FALSE"),
    ("asthma_mca", "asthma", 3, "TRUE"),
    ("hypercholesterolemia", "hypercholesterolemia", 3, "FALSE"),
    ("hypercholesterolemia_mca", "hypercholesterolemia", 3, "TRUE"),
    ("coronary_heart_disease", "coronary_heart_disease", 3, "FALSE"),
    ("coronary_heart_disease_mca", "coronary_heart_disease", 3, "TRUE"),
    ("ckd", "ckd", 3, "FALSE"),
    ("ckd_mca", "ckd", 3, "TRUE"),
    ("prostate_cancer", "prostate_cancer", 1, "FALSE"),
    ("prostate_cancer_mca", "prostate_cancer", 1, "TRUE"),
    ("breast_cancer", "breast_cancer", 2, "FALSE"),
    ("breast_cancer_mca", "breast_cancer", 2, "TRUE")
]:
    param.append({'--env FOLDERNAME' : folder_name,
                  '--env PHENOCOL' : phenocol,
                  '--env SEXFILTER' : sexfilter,
                  '--env MCA' : mca,
                  '--output-recursive OUTPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}",
                  '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/step0.log"})

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_STEP0 = f'{cur_dir}/param/SDoH_MCA_GWAS_step0.tsv'

param.to_csv(PARAMETER_FILENAME_STEP0, sep = '\t', index = False)

%env PARAMETER_FILENAME_STEP0={PARAMETER_FILENAME_STEP0}

## Script to obtain phenotype file

In [ ]:
%%writefile step0.sh

#!/bin/bash

set -o errexit
set -o nounset

cat <<EOF > ~/PHENOSCRIPT.R
#------------------START OF R-----------------------------------------------------------------

library(data.table)
library(tidyverse)
library(optparse)

#------------Command line arguments

option_list = list(
    make_option("--dir", type = "character", help = "Full path to master directory to which all gwases are saved"),
    make_option("--pca", type = "character", help = "Location of PCA file"),
    make_option("--sexfilter", type = "integer", help = "Include males(1) or females(2) or all(3)"),
    make_option("--phenofile", type = "character", help = "Location of phenotype file if casefile not available"),
    make_option("--phenocol", type = "character", help = "phenotype column in --phenofile"),
    make_option("--agesex", type = "character", help = "Location of age and sex .Rda file"),
    make_option("--mca", type = "logical", help = "Include MCA axis as covariates?")
)

opt_parser = OptionParser(option_list = option_list)
opt = parse_args(opt_parser)

#--------------------Load datasets

if (opt\$mca){
    columns_to_select <- c("person_id", opt\$phenocol, paste0("mca_axis_", 0:19))
} else {
    columns_to_select <- c("person_id", opt\$phenocol)
}

phenofile <- fread(opt\$phenofile) %>%
    select(all_of(columns_to_select)) %>%
    rename(id = "person_id", phenotype = opt\$phenocol) %>%
    mutate(phenotype = as.integer(phenotype))

# Genotyping principal components
pca <- fread(opt\$pca) %>%
  select(all_of(c("id", "inf", paste0("PC", 1:10))))

# Info on ages and sex
ages <- fread(opt\$agesex) %>%
  select(all_of(c("id", "sex", "age"))) %>%
  mutate(age = as.numeric(age),
         sex = as.numeric(sex))

#---------------------Make phenotype file

pheno <- pca %>%
  left_join(ages, by = "id") %>%
  left_join(phenofile, by = "id") %>%
  na.omit() %>%
  mutate(agesex = age*sex,
         age2 = age^2,
         age2sex = (age^2)*sex)

if (opt\$sexfilter == 3){
    pheno <- pheno %>% filter(sex %in% c(1, 2))
} else if (opt\$sexfilter == 1){
    pheno <- pheno %>% filter(sex == 1)
} else if (opt\$sexfilter == 2){
    pheno <- pheno %>% filter(sex == 2)
} else {
    stop("Specify valid value of --sexfilter")
}

# Save directory
save_dir <- paste0(opt\$dir)

if (!dir.exists(paste0(save_dir, "/pheno_file"))) {
  dir.create(paste0(save_dir, "/pheno_file"))
}

write.table(pheno,
            file = paste0(save_dir, "/pheno_file/phenotype.tsv"),
            quote=FALSE, sep='\t', row.names = FALSE, col.names = TRUE)

write.table(pheno\$id,
            file = paste0(save_dir, "/pheno_file/sampleID.txt"),
            quote = FALSE, row.names = FALSE, col.names = FALSE)

write.table(pheno\$id[pheno\$sex == 1],
            file = paste0(save_dir, "/pheno_file/sampleID_male.txt"),
            quote = FALSE, row.names = FALSE, col.names = FALSE)

for (anc in unique(pheno\$inf)){
  
  write.table(pheno\$id[pheno\$inf == anc],
              file = paste0(save_dir, "/pheno_file/", anc, "sampleID.txt"),
              quote = FALSE, row.names = FALSE, col.names = FALSE)
  
  # Male sample ID
  write.table(pheno\$id[(pheno\$inf == anc) & (pheno\$sex == 1)],
              file = paste0(save_dir, "/pheno_file/", anc, "sampleID_male.txt"),
              quote = FALSE, row.names = FALSE, col.names = FALSE)
  
}

# Display for output
cat("\nColumns included in phenotype file\n")
cat(paste(colnames(pheno), sep = ","))

cat("\nCase/Control\n")
table(pheno\$phenotype)

cat("\nCase/Control in each ancestry\n")
table(pheno\$inf, pheno\$phenotype)

cat("\nCase/Control among males in each ancestry\n")
table(pheno\$inf[pheno\$sex == 1], pheno\$phenotype[pheno\$sex == 1])

cat("\nCase/Control among females in each ancestry\n")
table(pheno\$inf[pheno\$sex == 2], pheno\$phenotype[pheno\$sex == 2])

#----------Ancestry with >20 cases

ancetsry_caseCounts <- c()

for (ancestry in unique(pheno\$inf)) {
  
  if (length(pheno\$id[(pheno\$inf == ancestry) & (pheno\$phenotype == 1)]) >= 20){
    ancetsry_caseCounts <- c(ancetsry_caseCounts, ancestry)
  }
  
}

write.table(ancetsry_caseCounts,
            file = paste0(save_dir, "/pheno_file/", "ancestry_for_meta.txt"),
            quote = FALSE, row.names = FALSE, col.names = FALSE)

#------------------END OF R-------------------------------------------------------------------
EOF

Rscript --vanilla \
    ~/PHENOSCRIPT.R \
    --dir "${OUTPATH}" \
    --pca "${PCA}" \
    --sexfilter "${SEXFILTER}" \
    --phenofile "${PHENOFILE}" \
    --phenocol "${PHENOCOL}" \
    --agesex "${AGESEX}" \
    --mca "${MCA}" > "${LOGFILE}"

In [ ]:
!gsutil mv step0.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step0.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

# --skip \

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/abiji/aou_step0:v2" \
    --min-ram 15 \
    --min-cores 4 \
    --disk-size 20 \
    --input PCA="${WORKSPACE_BUCKET}/data/meta/pca/pca_population_stratified.tsv" \
    --input PHENOFILE="gs://fc-secure-569df2b8-376c-4789-bcbf-09a0c417bd47/data/analysis_files/all_disease.tsv" \
    --input AGESEX="${WORKSPACE_BUCKET}/data/meta/phenotype_data/age_sex.tsv" \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/step0.log" \
    --tasks "${PARAMETER_FILENAME_STEP0}" \
    --preemptible \
    --retries 3 \
    --wait \
    --skip \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step0.sh"

# Function that returns ancestry with case >= 20

In [ ]:
def ancestry_include(folder_name, main = f"{MAIN_DIR}", saige = f"{SAIGE_DIR}"):
    path = subprocess.run(shlex.split(f"gsutil ls {main}/{saige}/{folder_name}/pheno_file/"),
                          capture_output = True,
                          text = True,
                          check = True)
    
    path = path.stdout

    codes = re.findall(r'/([A-Z]{3})sampleID\.txt', path)

    codes = [x.lower() for x in codes if 'oth' not in x.lower()]  # Exclude elements containing 'oth'
    codes = [x.lower() for x in codes if 'mid' not in x.lower()]  # Exclude elements containing 'mid'

    return(codes)

In [ ]:
ancestry_include("asthma_mca")

# Step 1

## Make parameter file for each ancestry

In [ ]:
# Initialize an Empty DataFrame
param = []

for folder_name, sex_filter in [
    ("asthma", "FALSE"),
    ("asthma_mca", "FALSE"),
    ("hypercholesterolemia", "FALSE"),
    ("hypercholesterolemia_mca", "FALSE"),
    ("coronary_heart_disease", "FALSE"),
    ("coronary_heart_disease_mca", "FALSE"),
    ("ckd", "FALSE"),
    ("ckd_mca", "FALSE"),
    ("prostate_cancer", "TRUE"),
    ("prostate_cancer_mca", "TRUE"),
    ("breast_cancer", "TRUE"),
    ("breast_cancer_mca", "TRUE")
]:
    for anc in ancestry_include(folder_name):
        param.append({
            '--env ancestry' : anc.upper(),
            '--env FOLDERNAME' : folder_name,
            '--env MCA' : "," + ",".join(f"mca_axis_{i}" for i in range(20)) if "_mca" in folder_name else "",
            '--env SEXFILTER' : sex_filter,
            '--input-recursive plinkfile' : f"{my_bucket}/data/meta/plink_array/qc_{anc}",
            '--input-recursive PHENOPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/pheno_file",
            '--input grm' : f"{my_bucket}/data/meta/saige_grm/{anc}_relatednessCutoff_0.125_5000_randomMarkersUsed.sparseGRM.mtx",
            '--input grm_id' : f"{my_bucket}/data/meta/saige_grm/{anc}_relatednessCutoff_0.125_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt",
            '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/step1_{anc}.log",
            '--output-recursive OUTPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step1"
        })

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_STEP1 = f'{cur_dir}/param/SDoH_MCA_GWAS_step1.tsv'

param.to_csv(PARAMETER_FILENAME_STEP1, sep = '\t', index = False)

%env PARAMETER_FILENAME_STEP1={PARAMETER_FILENAME_STEP1}

## Script to obtain step 1

In [ ]:
%%writefile step1.sh

#!/bin/bash

set -o errexit
set -o nounset

if [[ "${SEXFILTER}" == "FALSE" ]]; then
    step1_fitNULLGLMM.R \
        --plinkFile="${plinkfile}/plinkarray_qc_${ancestry,,}" \
        --sparseGRMFile="${grm}"   \
        --sparseGRMSampleIDFile="${grm_id}" \
        --useSparseGRMtoFitNULL=TRUE \
        --skipVarianceRatioEstimation=FALSE \
        --phenoFile="${PHENOPATH}/phenotype.tsv" \
        --SampleIDIncludeFile="${PHENOPATH}/${ancestry}sampleID.txt" \
        --outputPrefix="${OUTPATH}/${ancestry}saige_step1" \
        --phenoCol=phenotype \
        --covarColList=age,sex,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,age2,agesex,age2sex"${MCA}" \
        --sampleIDColinphenoFile=id \
        --traitType=binary \
        --IsOverwriteVarianceRatioFile=TRUE \
        --sexCol=sex \
        --MaleCode=1 \
        --FemaleCode=2 \
        --LOCO=FALSE \
        --qCovarColList=sex \
        --isCovariateOffset=FALSE > "${LOGFILE}"
elif [[ "${SEXFILTER}" == "TRUE" ]]; then
    step1_fitNULLGLMM.R \
        --plinkFile="${plinkfile}/plinkarray_qc_${ancestry,,}" \
        --sparseGRMFile="${grm}"   \
        --sparseGRMSampleIDFile="${grm_id}" \
        --useSparseGRMtoFitNULL=TRUE \
        --skipVarianceRatioEstimation=FALSE \
        --phenoFile="${PHENOPATH}/phenotype.tsv" \
        --SampleIDIncludeFile="${PHENOPATH}/${ancestry}sampleID.txt" \
        --outputPrefix="${OUTPATH}/${ancestry}saige_step1" \
        --phenoCol=phenotype \
        --covarColList=age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,age2"${MCA}" \
        --sampleIDColinphenoFile=id \
        --traitType=binary \
        --IsOverwriteVarianceRatioFile=TRUE \
        --LOCO=FALSE \
        --isCovariateOffset=FALSE > "${LOGFILE}"
else
    echo "ERROR: SEXFILTER must be 'TRUE' or 'FALSE' (got '${SEXFILTER}')." >&2
    exit 1
fi

In [ ]:
!gsutil mv step1.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step1.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step1.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.3.6" \
    --min-ram 16 \
    --min-cores 1 \
    --disk-size 100 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/step1.log" \
    --tasks "${PARAMETER_FILENAME_STEP1}" \
    --preemptible \
    --retries 3 \
    --wait \
    --skip \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step1.sh"

# Step 2

## Make parameter file for each ancestry

In [ ]:
# Initialize an Empty DataFrame
param = []

for folder_name in [
    "asthma",
    "asthma_mca",
    "hypercholesterolemia",
    "hypercholesterolemia_mca",
    "coronary_heart_disease",
    "coronary_heart_disease_mca",
    "ckd",
    "ckd_mca",
    "prostate_cancer",
    "prostate_cancer_mca",
    "breast_cancer",
    "breast_cancer_mca",
]:
    for anc in ancestry_include(folder_name):
        for chrom in range(1, 23): # Chromosomes 1 to 22 range(1, 23)
            param.append({
                '--input id' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/pheno_file/{anc.upper()}sampleID.txt",
                '--input bgen' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.bgen",
                '--input bgi' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.bgen.bgi",
                '--input sample' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.sample",
                '--input grm' : f"{my_bucket}/data/meta/saige_grm/{anc}_relatednessCutoff_0.125_5000_randomMarkersUsed.sparseGRM.mtx",
                '--input grm_id' : f"{my_bucket}/data/meta/saige_grm/{anc}_relatednessCutoff_0.125_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt",
                '--input gmmat' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step1/{anc.upper()}saige_step1.rda",
                '--input var_ratio' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step1/{anc.upper()}saige_step1.varianceRatio.txt",
                '--env ancestry' : anc.upper(),
                '--env FOLDERNAME' : folder_name,
                '--env chrom' : chrom,
                '--label chrom' : chrom,
                '--label ancestry': anc,
                '--output OUTFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step2/{anc.upper()}chr{chrom}saige_step2",
                '--output OUTINDEX' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step2/{anc.upper()}chr{chrom}saige_step2.index",
                '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/step2{anc.upper()}chr{chrom}.log"
            })

param = pd.DataFrame(param)
param.head()

In [ ]:
PARAMETER_FILENAME_STEP2 = f'{cur_dir}/param/SDoH_MCA_GWAS_step2.tsv'

param.to_csv(PARAMETER_FILENAME_STEP2, sep = '\t', index = False)

%env PARAMETER_FILENAME_STEP2={PARAMETER_FILENAME_STEP2}

## Script to obtain step 2

In [ ]:
%%writefile step2.sh

#!/bin/bash

set -o errexit
set -o nounset

# This script adds the sex column to bgen sample file
Rscript --vanilla \
    "${BGENFIX}" \
    --sex "${AGESEX}" \
    --sample "${sample}"

step2_SPAtests.R \
    --sampleFile="~/bgen.sample" \
    --bgenFile="${bgen}" \
    --bgenFileIndex="${bgi}" \
    --AlleleOrder=ref-first \
    --SAIGEOutputFile="${OUTFILE}" \
    --subSampleFile="${id}" \
    --sparseGRMFile="${grm}"   \
    --sparseGRMSampleIDFile="${grm_id}" \
    --chrom="chr${chrom}" \
    --varianceRatioFile="${var_ratio}" \
    --GMMATmodelFile="${gmmat}" \
    --relatednessCutoff=0.125 \
    --is_imputed_data=FALSE \
    --is_overwrite_output=TRUE \
    --is_Firth_beta=TRUE \
    --pCutoffforFirth=0.05 \
    --minMAC=200 \
    --LOCO=FALSE \
    --is_output_moreDetails=TRUE \
    --is_fastTest=TRUE > "${LOGFILE}"

In [ ]:
!gsutil mv step2.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step2.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step2.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.3.6" \
    --min-ram 6 \
    --min-cores 1 \
    --disk-size 800 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/step2.log" \
    --input BGENFIX="${WORKSPACE_BUCKET}/data/scripts/r/fix_bgenSampleFile.R" \
    --input AGESEX="${WORKSPACE_BUCKET}/data/meta/phenotype_data/age_sex.tsv" \
    --tasks "${PARAMETER_FILENAME_STEP2}" \
    --skip \
    --preemptible \
    --retries 2 \
    --wait \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/step2.sh"

# Combine output

## Make parameter file for each ancestry

In [ ]:
param = []

for folder_name in [
    "asthma",
    "asthma_mca",
    "hypercholesterolemia",
    "hypercholesterolemia_mca",
    "coronary_heart_disease",
    "coronary_heart_disease_mca",
    "ckd",
    "ckd_mca",
    "prostate_cancer",
    "prostate_cancer_mca",
    "breast_cancer",
    "breast_cancer_mca",
]:
    for anc in ancestry_include(folder_name):
        param.append({
            '--input res' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step2/{anc.upper()}chr*",
            '--output-recursive OUTPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}",
            '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/combine{anc.upper()}.log",
            '--env title' : folder_name,
            '--env ancestry' : anc.upper()
        })

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_COMBINE = f'{cur_dir}/param/SDoH_MCA_GWAS_combine.tsv'

param.to_csv(PARAMETER_FILENAME_COMBINE, sep = '\t', index = False)

%env PARAMETER_FILENAME_COMBINE={PARAMETER_FILENAME_COMBINE}

## Script to combine results

In [ ]:
%%writefile combine.sh

#!/bin/bash

set -o errexit
set -o nounset

cat <<EOF > ~/COMBINE_SCRIPT.R
#------------------START OF R-----------------------------------------------------------------

library(data.table)
library(tidyverse)
library(optparse)
library(qqman)

#------------Command line arguments---------------------------------------------

# Ref: https://www.r-bloggers.com/2015/09/passing-arguments-to-an-r-script-from-command-lines/

option_list = list(
  make_option("--indir", type = "character", default = "", help = "Required. Input dir",),
  make_option("--outdir", type = "character", default = "", help = "Required. Output dir",),
  make_option("--suffix", type = "character", default = "", help = "Required only when --metaanalysis = 1",),
  make_option("--title", type = "character", default = "", help = "Required. Title of plot",)
)

opt_parser = OptionParser(option_list = option_list)
opt = parse_args(opt_parser)

#----------Key function--------------------------------------------------------

generate_unique_key <- function(CHR, POS, REF, ALT) {

  # Handle missing cases
  if (any(is.na(CHR)) || any(is.na(POS)) || any(is.na(REF)) || any(is.na(ALT))) {
    stop("Missing values in CHR, POS, REF, ALT")
  }

  # Consider chr1 vs 1
  CHR <- gsub("chr", "", CHR)

  # Ensure REF and ALT are in consistent case
  REF <- toupper(REF)
  ALT <- toupper(ALT)

  # Sort REF and ALT lexicographically for all rows
  sorted_alleles <- pmap_chr(
    list(REF, ALT),
    ~ paste(sort(c(..1, ..2)), collapse = "-")
  )

  # Combine CHR, POS, and sorted alleles
  paste(CHR, POS, sorted_alleles, sep = "-")
}

#-----------Loading datasets

filenames_step2 <- list.files(paste0(opt\$indir), pattern = paste0(opt\$suffix, ".*2$"))
combined_step2 <- lapply(paste0(opt\$indir, "/", filenames_step2), fread)

#----------Merge and save

combined_step2 <- do.call(rbind, combined_step2) %>%
  mutate(unique_key = generate_unique_key(CHR, POS, Allele1, Allele2))

if (("N_ctrl" %in% colnames(combined_step2)) & ("N_case" %in% colnames(combined_step2))){
  combined_step2 <- combined_step2 %>%
    mutate(N = N_ctrl + N_case) %>%
    mutate(Neff = 4/((1/N_ctrl) + (1/N_case)))
}

if (!dir.exists(paste0(opt\$outdir, "/step2"))) {
  dir.create(paste0(opt\$outdir, "/step2"))
}

write.table(combined_step2,
            file = paste0(opt\$outdir, "/step2/", opt\$suffix, ".tsv"),
            quote=FALSE, sep='\t', row.names = FALSE, col.names = TRUE)

# Make individual Manhattan plots as well
combined_step2 <- combined_step2 %>%
  mutate(CHR = ifelse(CHR == "X", 23, gsub("chr", "", CHR))) %>%
  filter(AF_Allele2 > 0.005) %>%
  mutate(CHR = as.numeric(CHR))

if (!dir.exists(paste0(opt\$outdir, "/figs"))) {
  dir.create(paste0(opt\$outdir, "/figs"))
}

# Draw Manhattan plot
png(file=paste0(opt\$outdir, "/figs/", opt\$suffix, "_manhattan.png"), width=1200, height=600)
manhattan(combined_step2,
          bp='POS',
          snp='MarkerID',
          p='p.value',
          main=paste0(opt\$suffix, " Manhattan plot for\n", opt\$title))
dev.off()

# Draw Q-Q plot
png(file=paste0(opt\$outdir, "/figs/", opt\$suffix, "_qqplot.png"), width=600, height=600)
lambda <- median(qchisq(1 - combined_step2\$p.value,1))/qchisq(0.5,1)
qq(combined_step2\$p.value,
   main=paste0(opt\$suffix, " Q-Q plot (\u03BB = ", signif(lambda, digits = 3), ") for\n", opt\$title))
dev.off()


#-----------Export lambda

if (!dir.exists(paste0(opt\$outdir, "/additional_analysis"))) {
  dir.create(paste0(opt\$outdir, "/additional_analysis"))
}

write.table(lambda,
            file = paste0(opt\$outdir, "/additional_analysis/", opt\$suffix, "_lambda.txt"),
            quote = FALSE, sep='\t', row.names = FALSE, col.names = FALSE)

#------------------END OF R-------------------------------------------------------------------
EOF

INPUT_FILES_PATH="$(dirname "${res}")"

Rscript --vanilla \
    ~/COMBINE_SCRIPT.R \
    --indir "${INPUT_FILES_PATH}" \
    --outdir "${OUTPATH}" \
    --suffix "${ancestry}" \
    --title "${title}" > "${LOGFILE}"

In [ ]:
!gsutil mv combine.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/combine.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/combine.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/abiji/aou_step0:v2" \
    --min-ram 24 \
    --min-cores 4 \
    --disk-size 150 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/combine.log" \
    --tasks "${PARAMETER_FILENAME_COMBINE}" \
    --preemptible \
    --retries 2 \
    --wait \
    --skip \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/combine.sh"

# Meta-analysis

## Parameter

In [ ]:
param = []

for folder_name in [
    "asthma",
    "asthma_mca",
    "hypercholesterolemia",
    "hypercholesterolemia_mca",
    "coronary_heart_disease",
    "coronary_heart_disease_mca",
    "ckd",
    "ckd_mca",
    "prostate_cancer",
    "prostate_cancer_mca",
    "breast_cancer",
    "breast_cancer_mca",
]:
    param.append({
        '--input step2' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/step2/*.tsv",
        '--input ancestry_file' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/pheno_file/ancestry_for_meta.txt",
        '--env title' : folder_name,
        '--output-recursive OUTPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}",
        '--output METAL_LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/metal.log",
        '--output PLOT_LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/plot.log"
    })

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_META = f'{cur_dir}/param/SDoH_MCA_GWAS_meta.tsv'

param.to_csv(PARAMETER_FILENAME_META, sep = '\t', index = False)

%env PARAMETER_FILENAME_META={PARAMETER_FILENAME_META}

## Script to conduct meta-analysis and plot

In [ ]:
%%writefile meta.sh

#!/bin/bash

set -o errexit
set -o nounset

new_dir="${OUTPATH}/bash"
if [ ! -d "$new_dir" ]; then
    mkdir "$new_dir"
fi

meta=${OUTPATH}/bash/metal.sh
INPUT_FILES_PATH="$(dirname "${step2}")"

cat << EOF > $meta

SEPARATOR TAB
SCHEME STDERR
TRACKPOSITIONS ON
AVERAGEFREQ ON
MINMAXFREQ ON

MARKER MarkerID
WEIGHT N
ALLELE Allele2 Allele1
EFFECT BETA
STDERR SE
PVALUE p.value
FREQ AF_Allele2
CHROMOSOMELABEL CHR
POSITIONLABEL POS
ADDFILTER AF_Allele2 > 0.005

EOF

# Add each ancestry to the meta-analysis if they have enough case numbers
for anc_withCase in $(grep -Ev 'OTH|MID' "${ancestry_file}"); do
    cat << EOF >> $meta

PROCESS ${INPUT_FILES_PATH}/${anc_withCase}.tsv

EOF
done

# Conclude meta-analysis by specifying name of output file
cat << EOF >> $meta

OUTFILE ${OUTPATH}/${META_FILE_NAME} .tsv
ANALYZE HETEROGENEITY 
QUIT

EOF

metal "${meta}" > "${METAL_LOGFILE}"

#--------------------Plot-----------------------------------------

ANCESTRY_NAME="$(grep -Ev 'OTH|MID' ${ancestry_file} | paste -sd ',')"

Rscript --vanilla \
    "${PLOT_SCRIPT}" \
    --indir "${OUTPATH}" \
    --outdir "${OUTPATH}" \
    --ancestry "${ANCESTRY_NAME}" \
    --title "${title}" > "${PLOT_LOGFILE}"

#-----------------Forest plot-------------------------------------

if [[ -s "${OUTPATH}/peaks/meta_peaks.txt" ]]; then

ipynb="${OUTPATH}/bash/ipynb.sh"

cat << EOF >> $ipynb
#!/bin/bash

python3 "${FP_SCRIPT}" \\
    --notebook_name "${OUTPATH}/peaks/gwas_meta.ipynb" \\
EOF

for anc_withCase in $(grep -Ev 'OTH|MID' "${ancestry_file}"); do
cat << EOF >> $ipynb
    --sumstat CHR POS Allele1 Allele2 p.value "${INPUT_FILES_PATH}/${anc_withCase}.tsv" BETA SE AF_Allele2 N_case N_ctrl BioMe_${anc_withCase} \\
EOF
done
    
#------------Reads each line of variant file and adds them--------
while IFS= read -r line; do
cat << EOF >> $ipynb
    --variant $line \\
EOF
done < "${OUTPATH}/peaks/meta_peaks.txt"
#-----------------------------------------------------------------

cat << EOF >> $ipynb
    --plot_title "${title}"
EOF

chmod u+x ${OUTPATH}/bash/ipynb.sh
${OUTPATH}/bash/ipynb.sh
jupyter-nbconvert --execute --to notebook --ExecutePreprocessor.timeout=-1 --inplace "${OUTPATH}/peaks/gwas_meta.ipynb"
jupyter-nbconvert --to html "${OUTPATH}/peaks/gwas_meta.ipynb"

fi

In [ ]:
!gsutil mv meta.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/meta.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/meta.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/abiji/aou_downstream:v2" \
    --min-ram 15 \
    --min-cores 4 \
    --disk-size 150 \
    --env META_FILE_NAME="gwas_meta" \
    --input PLOT_SCRIPT="${WORKSPACE_BUCKET}/data/scripts/r/plot_and_getpeaks.R" \
    --input FP_SCRIPT="${WORKSPACE_BUCKET}/data/scripts/py/fpNotebook_general.py" \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/meta.log" \
    --tasks "${PARAMETER_FILENAME_META}" \
    --preemptible \
    --retries 2 \
    --wait \
    --skip \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/meta.sh"

# Figures for publication

In [ ]:
param = []

for folder_name in [
    "asthma",
    "asthma_mca",
    "hypercholesterolemia",
    "hypercholesterolemia_mca",
    "coronary_heart_disease",
    "coronary_heart_disease_mca",
    "ckd",
    "ckd_mca",
    "prostate_cancer",
    "prostate_cancer_mca",
    "breast_cancer",
    "breast_cancer_mca",
]:
    param.append({
        '--input sumstats' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/gwas_meta1.tsv",
        '--input ancestry_file' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/pheno_file/ancestry_for_meta.txt",
        '--env label' : folder_name,
        '--output OUTMAN' : f"{MAIN_DIR}/{SAIGE_DIR}/figs/{folder_name}_manhattan.tiff",
        '--output OUTQQ' : f"{MAIN_DIR}/{SAIGE_DIR}/figs/{folder_name}_qq.tiff"
    })

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_CUSTOM_FIGS = f'{cur_dir}/param/SDoH_MCA_GWAS_figs_for_paper.tsv'

param.to_csv(PARAMETER_FILENAME_CUSTOM_FIGS, sep = '\t', index = False)

%env PARAMETER_FILENAME_CUSTOM_FIGS={PARAMETER_FILENAME_CUSTOM_FIGS}

In [ ]:
%%writefile figs_for_paper.sh

#!/bin/bash

set -o errexit
set -o nounset

cat <<EOF > ~/ADD_MARKERID.R
#------------------START OF R-----------------------------------------------------------------

library(data.table)
library(tidyverse)
library(qqman)

sumstats <- fread("${sumstats}") %>%
    rename(P = "P-value") %>%
    mutate(Chromosome = gsub('chr', '', Chromosome)) %>%
    mutate(Chromosome = ifelse(Chromosome == "X", 23, Chromosome)) %>%
    mutate(Position = as.numeric(Position), Chromosome = as.numeric(Chromosome), P = as.numeric(P)) %>%
    filter(Chromosome %in% 1:22)

tiff(file = "${OUTMAN}", units="in", width=6, height=3, compression = 'lzw', res=600)

par(mar = c(5, 5, 4, 2) + 0.1)

manhattan(sumstats,
          #main = "${label}",
          chr = "Chromosome",
          bp = 'Position',
          snp = 'MarkerName',
          p = 'P',
          col = c("darkblue", "#2596be"),
          ylim = c(0, max(-log10(sumstats\$P)) + 1),
          cex = 1,
          cex.axis = 1,
          cex.lab  = 1
         )

dev.off()

lambda <- median(qchisq(1 - sumstats\$P, 1)) / qchisq(0.5, 1)

tiff(file = "${OUTQQ}", units="in", width=3, height=3, compression = 'lzw', res=600)

par(mar = c(5, 5, 4, 2) + 0.1)

qq(sumstats\$P,
   main = paste0("\u03BB = ", signif(lambda, digits = 3)),
   cex = 1,
   cex.axis = 1,
   cex.lab  = 1
  )

dev.off()

#------------------END OF R-------------------------------------------------------------------
EOF

Rscript --vanilla ~/ADD_MARKERID.R

In [ ]:
!gsutil mv figs_for_paper.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/figs_for_paper.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/figs_for_paper.sh

In [ ]:
#! gsutil rm -r ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/figs/logs

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/abiji/aou_downstream:v2" \
    --min-ram 32 \
    --min-cores 8 \
    --disk-size 50 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/figs/logs/manhattan_qq.log" \
    --preemptible \
    --retries 2 \
    --wait \
    --skip \
    --tasks "${PARAMETER_FILENAME_CUSTOM_FIGS}" \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/figs_for_paper.sh"

# Extract peak details

## Parameter

In [ ]:
param = []

for folder_name in [
    "asthma",
    "hypercholesterolemia",
    "coronary_heart_disease",
    "ckd",
    "prostate_cancer",
    "breast_cancer",
]:
    
    peak_file_path = f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/peaks/meta_peaks.txt"
    peak_file_path_mca = f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}_mca/peaks/meta_peaks.txt"
    
    # Check if the file exists in GCS
    result = subprocess.run(
        ["gsutil", "-q", "stat", peak_file_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    
    result_mca = subprocess.run(
        ["gsutil", "-q", "stat", peak_file_path_mca],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    if result.returncode != 0 and result_mca.returncode != 0:
        print(f"Skipping {phe} — does not exist.")
        continue
        
    param.append({
        '--input sumstats' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/gwas_meta1.tsv",
        '--input sumstats_mca' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}_mca/gwas_meta1.tsv",
        '--input peak' : peak_file_path,
        '--input peak_mca' : peak_file_path_mca,
        '--env label' : folder_name,
        '--output OUTFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/peaks/details_{folder_name}.tsv",
        '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/{folder_name}/logs/peakdetails_META.log"
    })

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME_EXTRACT_PEAKS = f'{cur_dir}/param/SDoH_MCA_GWAS_extract_peak_details.tsv'

param.to_csv(PARAMETER_FILENAME_EXTRACT_PEAKS, sep = '\t', index = False)

%env PARAMETER_FILENAME_EXTRACT_PEAKS={PARAMETER_FILENAME_EXTRACT_PEAKS}

## Script

In [ ]:
%%writefile get_peak_details.sh

#!/bin/bash

set -o errexit
set -o nounset

cat <<EOF > ~/GET_PEAK_DETAILS.R
#------------------START OF R-----------------------------------------------------------------

library(tidyverse)
library(data.table)

#-------------------------------------------------------------------------------

# # Uncomment for p < 5e-8 filtering
# get_peaks <- function(meta_file){
  
#   meta_filtered <- meta_file %>%
#     mutate(Allele1 = toupper(Allele1), Allele2 = toupper(Allele2))# %>%
#     #filter(p_value < 5*10^(-8))
  
#   peaks <- data.frame()
  
#   while (nrow(meta_filtered) > 0) {
    
#     min_p <- meta_filtered %>%
#       filter(p_value == min(p_value, na.rm = TRUE))
    
#     # Remove the first peak and its neighborhood
#     meta_filtered <- meta_filtered %>%
#       filter(!(
#           (Chromosome == min_p\$Chromosome) &
#           (Position > min_p\$Position - 500000) &
#           (Position < min_p\$Position + 500000)
#       ))
    
#     peaks <- rbind(peaks, min_p)
    
#   }
  
#   return(peaks)
  
# }

#-------------------------------------------------------------------------------

varfile <- fread("${peak}", header = FALSE) %>%
    magrittr::set_colnames(c("chromosome", "position", "ref", "alt")) %>%
    mutate(
        marker = paste(paste0("chr", chromosome), position, toupper(ref), toupper(alt), sep = ":"),
        marker_rev = paste(paste0("chr", chromosome), position, toupper(alt), toupper(ref), sep = ":")
    )

varfile_mca <- fread("${peak_mca}", header = FALSE) %>%
    magrittr::set_colnames(c("chromosome", "position", "ref", "alt")) %>%
    mutate(
        marker = paste(paste0("chr", chromosome), position, toupper(ref), toupper(alt), sep = ":"),
        marker_rev = paste(paste0("chr", chromosome), position, toupper(alt), toupper(ref), sep = ":")
    )

peak_markers <- unique(c(
    varfile\$marker,
    varfile\$marker_rev,
    varfile_mca\$marker,
    varfile_mca\$marker_rev
))

sumstats <- fread("${sumstats}", header = TRUE) %>%
    filter(MarkerName %in% peak_markers) %>%
    mutate(phenotype = "${label}", mca = "FALSE") %>%
    rename(p_value = "P-value")
    
sumstats_peaks <- sumstats#get_peaks(sumstats)

sumstats_mca <- fread("${sumstats_mca}", header = TRUE) %>%
    filter(MarkerName %in% peak_markers) %>%
    mutate(phenotype = "${label}", mca = "TRUE") %>%
    rename(p_value = "P-value")

sumstats_mca_peaks <- sumstats_mca #get_peaks(sumstats_mca)

final_peak_file <- data.frame()

if ((nrow(sumstats_peaks)>0) | (nrow(sumstats_mca_peaks)>0)){
    
    final_peak_file <- rbind(sumstats_peaks, sumstats_mca_peaks)
    
}

write.table(final_peak_file,
            file = "${OUTFILE}",
            quote=FALSE, sep='\t', row.names = FALSE, col.names = TRUE)

#------------------END OF R-------------------------------------------------------------------
EOF

Rscript --vanilla ~/GET_PEAK_DETAILS.R >> "${LOGFILE}"

In [ ]:
!gsutil mv get_peak_details.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/get_peak_details.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/get_peak_details.sh

## Submit dsub

In [ ]:
#! gsutil rm -r gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/meta/SDoH_MCA_GWAS/figs/logs

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/abiji/aou_step0:v2" \
    --min-ram 32 \
    --min-cores 8 \
    --disk-size 50 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/{job-name}/{job-id}-{task-id}-{task-attempt}.log" \
    --preemptible \
    --retries 2 \
    --wait \
    --skip \
    --tasks "${PARAMETER_FILENAME_EXTRACT_PEAKS}" \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/get_peak_details.sh"

# Extract PED files for peaks

## Make parameter file for each chromosome

In [ ]:
def unique_var_from_peak_files(gcs_folder):
    
    ls_result = subprocess.run(
        ["gsutil", "ls", f"{gcs_folder}/*.tsv"],
        check=True,
        capture_output=True,
        text=True
    )
    
    tsv_files = ls_result.stdout.strip().split("\n")

    dfs = []

    for file_path in tsv_files:
        cat_result = subprocess.run(
            ["gsutil", "cat", file_path],
            check=True,
            capture_output=True,
            text=True
        )

        dfs.append(pd.read_csv(
            StringIO(cat_result.stdout),
            sep="\t",
            usecols=["Chromosome", "MarkerName"]
        ))

    # Combine all files
    combined_df = pd.concat(dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
    
    return combined_df

In [ ]:
all_variants = unique_var_from_peak_files(f"{MAIN_DIR}/{SAIGE_DIR}/peaks")

In [ ]:
print(all_variants.shape)
all_variants.head()

In [ ]:
param = []

for chrom in all_variants["Chromosome"].unique():
    
    subset = all_variants.loc[all_variants["Chromosome"] == chrom, "MarkerName"].dropna().unique()
    marker_string = "\\n".join(subset)
    
    # Remove the leading chr from chromosome
    chrom = chrom.replace('chr', '')
    
    param.append({
        '--input bgen' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.bgen",
        '--input bgi' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.bgen.bgi",
        '--input sample' : f"gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/bgen/acaf_threshold.chr{chrom}.sample",
        '--env chrom' : chrom,
        '--env MARKER' : marker_string,
        '--output-recursive OUTPATH' : f"{MAIN_DIR}/{SAIGE_DIR}/peaks_ped",
        '--output LOGFILE' : f"{MAIN_DIR}/{SAIGE_DIR}/peaks_ped/logs/chr{chrom}.log"
    })

param = pd.DataFrame(param)
print(param.shape)
param.head()

In [ ]:
PARAMETER_FILENAME_PED = f'{cur_dir}/param/SDoH_MCA_GWAS_ped.tsv'

param.to_csv(PARAMETER_FILENAME_PED, sep = '\t', index = False)

%env PARAMETER_FILENAME_PED={PARAMETER_FILENAME_PED}

## Script

In [ ]:
%%writefile extract_ped.sh

#!/bin/bash

set -o errexit
set -o nounset

printf '%b\n' "${MARKER}" > ~/variants.txt

cat ~/variants.txt

cat ~/variants.txt > "${LOGFILE}"

plink2 \
    --bgen "${bgen}" ref-first \
    --sample "${sample}" \
    --extract ~/variants.txt \
    --export A \
    --memory 13500 \
    --out "${OUTPATH}/chr${chrom}" >> "${LOGFILE}"

In [ ]:
!gsutil mv extract_ped.sh ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/extract_ped.sh
!gsutil cat ${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/extract_ped.sh

## Submit bsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

#     --skip \
#     --preemptible \
#     --retries 5 \
#     --wait \
    
aou_dsub \
    --image "${ARTIFACT_REGISTRY_DOCKER_REPO}/pgscatalog/plink2:2.00a5.10" \
    --min-ram 16 \
    --min-cores 4 \
    --disk-size 800 \
    --logging "${MAIN_DIR}/${SAIGE_DIR}/logs/{job-name}/{job-id}-{task-id}-{task-attempt}.log" \
    --tasks "${PARAMETER_FILENAME_PED}" \
    --skip \
    --preemptible \
    --retries 1 \
    --wait \
    --script "${WORKSPACE_BUCKET}/data/${SAIGE_DIR}/bash/extract_ped.sh"